# ATML PA3 — Task 2 VLM (features, 3-stage curriculum, final evaluation)

GPU T4 x2, Internet on, attach `atml-pa3-code`, then *Save & Run All (Commit)*.
GPU 0 trains the main 196-token model; GPU 1 trains the optional 49-token variant at no
extra Kaggle-hour cost (set `RUN_49 = False` to skip). If cut off, attach this notebook's
previous output and rerun: features and checkpoints are restored and training resumes.

In [ ]:
# --- locate code (Kaggle dataset "atml-pa3-code" containing atml-pa3-code.zip, or Colab Drive) ---
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content/drive/MyDrive/atml-pa3"
if not ON_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
REPO = "/tmp/ATML-PA3"           # code lives outside the saved output
zips = glob.glob("/kaggle/input/**/atml-pa3-code.zip", recursive=True) + glob.glob(f"{WORK}/atml-pa3-code.zip")
# Kaggle usually unpacks an uploaded zip, so also accept an already-extracted copy
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/task1/train.py", recursive=True)]
shutil.rmtree(REPO, ignore_errors=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    assert dirs, "upload outputs/atml-pa3-code.zip as a Kaggle dataset (or to Drive/atml-pa3/)"
    shutil.copytree(dirs[0], REPO)
print("code from", zips[:1] or dirs[:1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "hf_xet"], check=True)
os.chdir(REPO)
os.makedirs(f"{WORK}/outputs", exist_ok=True)
if not os.path.exists("outputs"):
    os.symlink(f"{WORK}/outputs", "outputs")
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
RUN_49 = True
for prev in glob.glob("/kaggle/input/*/outputs/task2") if ON_KAGGLE else []:
    for item in os.listdir(prev):
        dst = f"outputs/task2/{item}"
        if not os.path.exists(dst):
            print("restoring", item); shutil.copytree(os.path.join(prev, item), dst)
if not os.path.exists("outputs/task2/features/info.json"):
    subprocess.run([sys.executable, "-m", "task2.features"], check=True)
print(open("outputs/task2/features/info.json").read())

In [ ]:
import time
runs = {"run196": ["--n_vis", "196"]}
if RUN_49 and torch.cuda.device_count() > 1:
    runs["run49"] = ["--n_vis", "49"]
procs = {}
for gpu, (name, extra) in enumerate(runs.items()):
    os.makedirs(f"outputs/task2/{name}", exist_ok=True)
    log = open(f"outputs/task2/{name}/stdout.log", "a")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    procs[name] = subprocess.Popen([sys.executable, "-m", "task2.train", "--out", f"outputs/task2/{name}",
                                    "--time_limit_h", "10.5", *extra], stdout=log, stderr=subprocess.STDOUT, env=env)
while any(p.poll() is None for p in procs.values()):
    time.sleep(180)
    for name in procs:
        tail = open(f"outputs/task2/{name}/stdout.log").read().strip().splitlines()[-1:]
        print(time.strftime("%H:%M"), name, tail[0][:300] if tail else "(starting)", flush=True)
print({n: p.returncode for n, p in procs.items()})

In [ ]:
for name, extra in runs.items():
    if os.path.exists(f"outputs/task2/{name}/stage3_vqa.safetensors"):
        subprocess.run([sys.executable, "-m", "task2.evaluate", "--run", f"outputs/task2/{name}",
                        "--out", f"outputs/task2/results/{name}", *extra], check=True)
!du -sh outputs/task2/*